# Month-over-Month Revenue Change Analysis

Given a table 'sf_transactions' of purchases by date, calculate the month-over-month percentage change in revenue. The output should include the year-month date (YYYY-MM) and percentage change, rounded to the 2nd decimal point, and sorted from the beginning of the year to the end of the year.

The percentage change column will be populated from the 2nd month forward and calculated as 
((this month’s revenue — last month’s revenue) / last month’s revenue)*100.


In [0]:
%skip
CREATE TABLE ska_catalog.bronze.transactions(id INT, created_at TIMESTAMP, value INT, purchase_id INT);

INSERT INTO ska_catalog.bronze.transactions VALUES
(1, '2019-01-01 00:00:00',  172692, 43), (2,'2019-01-05 00:00:00',  177194, 36),(3, '2019-01-09 00:00:00',  109513, 30),(4, '2019-01-13 00:00:00',  164911, 30),(5, '2019-01-17 00:00:00',  198872, 39), (6, '2019-01-21 00:00:00',  184853, 31),(7, '2019-01-25 00:00:00',  186817, 26), (8, '2019-01-29 00:00:00',  137784, 22),(9, '2019-02-02 00:00:00',  140032, 25), (10, '2019-02-06 00:00:00', 116948, 43), (11, '2019-02-10 00:00:00', 162515, 25), (12, '2019-02-14 00:00:00', 114256, 12), (13, '2019-02-18 00:00:00', 197465, 48), (14, '2019-02-22 00:00:00', 120741, 20), (15, '2019-02-26 00:00:00', 100074, 49), (16, '2019-03-02 00:00:00', 157548, 19), (17, '2019-03-06 00:00:00', 105506, 16), (18, '2019-03-10 00:00:00', 189351, 46), (19, '2019-03-14 00:00:00', 191231, 29), (20, '2019-03-18 00:00:00', 120575, 44), (21, '2019-03-22 00:00:00', 151688, 47), (22, '2019-03-26 00:00:00', 102327, 18), (23, '2019-03-30 00:00:00', 156147, 25);

In [0]:
SELECT * FROM ska_catalog.bronze.transactions;

### 𝐄𝐱𝐩𝐥𝐚𝐧𝐚𝐭𝐢𝐨𝐧 𝐨𝐟 𝐭𝐡𝐞 𝐐𝐮𝐞𝐫𝐲:
### 1. MonthlyRevenue CTE:
Aggregates the total revenue for each month using FORMAT to convert the created_at date to the format YYYY-MM.

### 2. RevenueChange CTE:
Adds a column previous_revenue using the LAG function, which fetches the total revenue of the previous month for each row.

### 3. Final SELECT:
Calculates the percentage change as ((total_revenue - previous_revenue) / previous_revenue) * 100. The ROUND function ensures the percentage is rounded to two decimal places. The output is ordered by year_month to display the data chronologically.

In [0]:
with month_revenue AS (
  SELECT date_format(created_at, 'yyyy-MM') AS `year_month`, SUM(value) AS `total_revenue`
  FROM ska_catalog.bronze.transactions
  GROUP BY year_month
),
previous_month AS (
  SELECT *, LAG(total_revenue) OVER (ORDER BY year_month) AS previous_revenue
  FROM month_revenue
)
SELECT year_month, total_revenue,
        CASE WHEN previous_revenue IS NULL THEN NULL
        ELSE ROUND(((total_revenue - previous_revenue)/CAST( previous_revenue AS FLOAT)) * 100, 4)
        END AS `percentage change`
FROM previous_month
ORDER BY  year_month

In [0]:
-- Retrieve all transactions made in February 2019.
SELECT id AS `TRANSACTION_ID`,
      YEAR(created_at) AS `YEAR`,
      MONTH(created_at) AS `MONTH`,
      DAY(created_at) AS `DATE`,
      UPPER(date_format(created_at, 'MMMM') )AS `MONTH_NAME`,
      value AS `AMOUNT`
FROM ska_catalog.bronze.transactions
WHERE MONTH(created_at) = 2

In [0]:
SELECT COUNT(id) AS `Total Transactions`, SUM(value) AS `Total value`  FROM ska_catalog.bronze.transactions

In [0]:
SELECT * FROM ska_catalog.bronze.transactions

In [0]:
-- Get the maximum, minimum, and average transaction value.
SELECT MAX(value) AS MAX_VALUE,
        MIN(value) AS MIN_VALUE,
        ROUND(AVG(value), 2) AS AVG_VALUE
FROM ska_catalog.bronze.transactions;

In [0]:
--List all distinct purchase IDs.
SELECT DISTINCT * FROM ska_catalog.bronze.transactions;

In [0]:
-- Find transactions where the value is greater than 180000.
SELECT id,created_at,value,purchase_id FROM ska_catalog.bronze.transactions
WHERE value > 180000
ORDER BY id;

In [0]:
-- Group transactions by purchase ID and calculate the total value per group.

SELECT purchase_id , SUM(value) AS `total_value` 
FROM ska_catalog.bronze.transactions
GROUP BY purchase_id
ORDER BY purchase_id;

In [0]:
-- Find the top 3 transactions with the highest value using rank function
SELECT * FROM
(
  SELECT *,
    DENSE_RANK() OVER (ORDER BY value DESC) AS rank 
  FROM ska_catalog.bronze.transactions
)
WHERE rank <= 3;

In [0]:
-- Calculate the number of transactions per month.
SELECT date_format(created_at, 'MMMM') AS `MONTH`, COUNT(id) AS `No of Transactions`
FROM ska_catalog.bronze.transactions
GROUP BY MONTH(created_at), MONTH
ORDER BY MONTH(created_at);

In [0]:
-- Find the average transaction value for each purchase ID.

SELECT purchase_id AS PURCHASE_ID,
        ROUND(AVG(value),2) AS average_value
FROM ska_catalog.bronze.transactions
GROUP BY PURCHASE_ID
ORDER BY PURCHASE_ID;

In [0]:
-- Identify transactions that occurred on weekends.
SELECT *,DAYOFWEEK(created_at) AS `Day_No`, date_format(created_at, 'EEEE') AS  `Day` 
FROM ska_catalog.bronze.transactions
WHERE date_format(DATE(created_at), 'EEEE') IN ('Sunday', 'Saturday')
ORDER BY DAYOFWEEK(created_at), Day ;